# Validation of Interpretability Methods in Cell Lines

In [ ]:
from __future__ import annotations

import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import pickle
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from scipy.spatial.distance import squareform, pdist
from statsmodels.stats.multitest import multipletests
from tensorflow import keras
from tqdm import tqdm

from cdrpy.util.io import read_gmt
from cdrpy.datasets import Dataset

from screendl import model as screendl
from screendl.utils.ensemble import ScreenDLEnsembleWrapper

from utils.plot import configure_chart
from utils.shap import shap_adapter_batch, simple_gsea, _agg_shap_values

In [ ]:
# whether or not to consider drugs without known response
CONSIDER_SCREENED_DRUGS_ONLY = True

# SHAP params
SHAP_ZSCORE_THRESHOLD = -1.5  # z-score threshold to consider a gene for GSEA
SHAP_NUM_BG_TUMORS = 20  # number of tumors in the SHAP background distribution
SHAP_NUM_BG_SAMPLES = 10000  # number of samples from background distribution

# number of reference tumors to consider
NUM_REFERENCE_TUMORS = 5

In [ ]:
root = Path("../../../datastore")

In [ ]:
# load MSigDB gene sets
gmt_files = [
    root / "raw/MSigDB/c6.all.v2023.2.Hs.symbols.gmt",
]

GENE_SETS = {}
for gmt_file in gmt_files:
    GENE_SETS.update(read_gmt(gmt_file))

print(f"Considering {len(GENE_SETS):,} gene sets for overrepresentation analysis.")

In [ ]:
dataset_dir = root / "inputs/CellModelPassports-GDSCv1v2-HCI-v2.0.0"

cell_meta = pd.read_csv(dataset_dir / "MetaSampleAnnotations.csv", index_col=0)
drug_meta = pd.read_csv(dataset_dir / "MetaDrugAnnotations.csv", index_col=0)

drug_encoders = screendl.load_drug_features(
    dataset_dir / "ScreenDL/FeatureMorganFingerprints.csv"
)
cell_encoders = screendl.load_cell_features(
    dataset_dir / "ScreenDL/FeatureGeneExpression.csv"
)

D = Dataset.from_csv(
    dataset_dir / "LabelsLogIC50.csv",
    cell_encoders=cell_encoders,
    drug_encoders=drug_encoders,
    cell_meta=cell_meta,
    drug_meta=drug_meta,
    name="CellModelPassports-GDSC-HCI",
)

cell_ids = set(D.obs[D.obs["cell_id"].str.startswith("SIDM")]["cell_id"])
D = D.select_cells(cell_ids, name="cell_ds")

print(D)

In [ ]:
def collapse_correlated_gene_sets(
    corr_mat: pd.DataFrame,
    threshold: float = 0.7,
) -> dict[str, list[str]]:
    """Collapse gene sets with high expression correlation."""
    dist = 1 - corr_mat.abs().values
    np.fill_diagonal(dist, 0)
    condensed = squareform(dist)
    Z = linkage(condensed, method="average")
    labels = fcluster(Z, t=1 - threshold, criterion="distance")

    clusters = {}
    for name, label in zip(corr_mat.columns, labels):
        clusters.setdefault(label, []).append(name)

    # representative
    result = {}
    for members in clusters.values():
        representative = sorted(members)[0]
        result[representative] = members

    return result

In [ ]:
def load_ensemble(root: Path) -> ScreenDLEnsembleWrapper:
    """Load the ScreenDL ensemble model."""
    members = []
    for file in root.glob("*/models/base.keras"):
        members.append(keras.models.load_model(file))

    return ScreenDLEnsembleWrapper(members)


def copy_dataset(D: Dataset) -> Dataset:
    """Create a deep copy of a Dataset."""
    obs = D.obs.copy(deep=True)

    cell_encoders = None
    if D.cell_encoders is not None:
        cell_encoders = {k: v.copy() for k, v in D.cell_encoders.items()}

    drug_encoders = None
    if D.drug_encoders is not None:
        drug_encoders = {k: v.copy() for k, v in D.drug_encoders.items()}

    return Dataset(
        obs,
        cell_encoders=cell_encoders,
        drug_encoders=drug_encoders,
        name=D.name,
        desc=D.desc,
    )


def preprocess_datasets(root: Path, D: Dataset) -> list[Dataset]:
    datasets = []
    for file in root.glob("*/preprocessing.pkl"):
        D_copy = copy_dataset(D)
        with open(file, "rb") as fh:
            proc = pickle.load(fh)
        D_copy.obs["label"] = (
            proc.response_scalers["cell"]
            .transform(D.obs[["label"]], groups=D.obs["drug_id"])
            .flatten()
        )
        D_copy.cell_encoders["exp"].data.loc[:, :] = proc.feature_scalers[
            "exp"
        ].transform(D_copy.cell_encoders["exp"].data.values)
        datasets.append(D_copy)

    return datasets

In [ ]:
dataset = "CellModelPassports-GDSCv1v2-HCI"
date = "2026-05-24_14-25-11"
path_fmt = "outputs/experiments/pdxo_validation/{0}/ScreenDL/multiruns/{1}"

run_dir = root / path_fmt.format(dataset, date)
model = load_ensemble(run_dir)
datasets = preprocess_datasets(run_dir, D)

In [ ]:
drop_ctypes = [
    "B-Lymphoblastic Leukemia",
    "B-Cell Non-Hodgkin's Lymphoma",
    "Acute Myeloid Leukemia",
    "T-Lymphoblastic Leukemia",
    "Plasma Cell Myeloma",
    "Burkitt's Lymphoma",
    "Chronic Myelogenous Leukemia",
    "Other Blood Cancers",
    "T-Cell Non-Hodgkin's Lymphoma",
    "Hodgkin's Lymphoma",
    "Melanoma",
    "Mesothelioma",
    "Glioblastoma",
    "Ewing's Sarcoma",
    "Neuroblastoma",
    "Small Cell Lung Carcinoma",
    "Other Solid Cancers",
]

keep_cancers = D.cell_meta.query("cancer_type not in @drop_ctypes").index.to_list()

In [ ]:
D.cell_encoders["exp"].data = D.cell_encoders["exp"].data.loc[
    list(cell_ids.intersection(keep_cancers))
]

gexp = D.cell_encoders["exp"].data.transform(stats.zscore).copy()

avg_gexp = []
for name, genes in GENE_SETS.items():
    avg_gexp.append(gexp.filter(items=genes).mean(axis=1).to_frame(name))

avg_gexp = pd.concat(avg_gexp, axis=1)
gs_corrs = avg_gexp.corr()

collapsed = collapse_correlated_gene_sets(gs_corrs, threshold=0.6)
print(f"Number of gene sets: {len(collapsed)}")

In [ ]:
def _collect_cell_ids(datasets: list[Dataset]) -> set[str]:
    cell_ids = set()
    for d in datasets:
        cell_ids.update(d.cell_ids)
    return cell_ids


def _annotate_gsea_results(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["fdr"] = multipletests(df["pval"], method="fdr_bh")[1]
    df["log_pval"] = -np.log10(df["pval"])
    df["log_fdr"] = -np.log10(df["fdr"])
    return df


def run_drug(
    drug_id: str,
    ensemble: ScreenDLEnsembleWrapper,
    datasets: list[Dataset],
    sets: dict[str, list[str]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run SHAP GSEA analysis for drug."""
    shap_func = lambda x: shap_adapter_batch(
        x[0],
        x[1].select_cells(keep_cancers),
        drug_id=drug_id,
        tumor_ids=None,
        n_bg_samples=SHAP_NUM_BG_TUMORS,
        n_shap_samples=SHAP_NUM_BG_SAMPLES,
        sorted_tumors=None,
        batch_size=32,
    )

    # we run SHAP for each ensemble member
    shap_values = list(map(shap_func, zip(ensemble.members, datasets)))
    cell_ids = _collect_cell_ids(datasets)

    gene_sets = {name: set(genes) for name, genes in sets.items()}

    all_shap_results = []
    all_gsea_results = []

    for cell_id in tqdm(cell_ids, desc="GSEA"):
        shap_for_t = list(filter(None, [x.get(cell_id) for x in shap_values]))
        t_shap_for_t = [x[0].assign(idx=i) for i, x in enumerate(shap_for_t)]

        if not t_shap_for_t:
            continue

        t_shap_for_t = pd.concat(t_shap_for_t, ignore_index=True)
        t_shap_for_t_agg = _agg_shap_values(t_shap_for_t)

        all_shap_results.append(
            t_shap_for_t_agg.assign(
                drug_id=drug_id,
                tumor_id=cell_id,
            )
        )

        # determine how many most-negative SHAP genes pass the z-score threshold.
        shap_zscores = stats.zscore(t_shap_for_t_agg["value"], ddof=0)
        passing = t_shap_for_t_agg.loc[shap_zscores < SHAP_ZSCORE_THRESHOLD]
        top_n_genes = passing.shape[0]

        if top_n_genes == 0:
            continue

        gene_shap = (
            t_shap_for_t_agg[["feat", "value"]]
            .sort_values("value", ascending=True)
            .values.tolist()
        )

        res = []

        for name, genes in gene_sets.items():
            p, overlap, M, K_eff, used_top_n = simple_gsea(
                gene_shap,
                genes,
                top_n=top_n_genes,
                by_abs=False,
                reversed=False,
            )

            total = t_shap_for_t_agg.loc[
                t_shap_for_t_agg["feat"].isin(genes),
                "value",
            ].sum()

            res.append(
                {
                    "drug_id": drug_id,
                    "tumor_id": cell_id,
                    "set": name,
                    "pval": p,
                    "overlap": overlap,
                    "M": M,
                    "K_eff": K_eff,
                    "top_n": used_top_n,
                    "threshold_top_n": top_n_genes,
                    "value": total,
                }
            )

        all_gsea_results.append(pd.DataFrame(res))

    all_shap_results_df: pd.DataFrame = pd.concat(
        all_shap_results,
        ignore_index=True,
    )

    all_gsea_results_df: pd.DataFrame = pd.concat(
        all_gsea_results,
        ignore_index=True,
    )

    all_gsea_results_df = (
        all_gsea_results_df.groupby(["drug_id", "tumor_id"], group_keys=False)
        .apply(_annotate_gsea_results)
        .reset_index(drop=True)
    )

    return all_gsea_results_df, all_shap_results_df

In [ ]:
TARGET_SETS = {k: GENE_SETS[k] for k in collapsed.keys()}
len(TARGET_SETS)

In [ ]:
target_drugs = [
    "Erlotinib",
    "Trametinib",
    "Lapatinib",
    "Cisplatin",
    "Dabrafenib",
    "AZD5363",
]

drug_results: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}
for drug_id in target_drugs:
    print(f"Running {drug_id}...")
    drug_results[drug_id] = run_drug(drug_id, model, datasets, TARGET_SETS)

In [ ]:
all_gsea_results = pd.concat([v[0].assign(drug_id=k) for k, v in drug_results.items()])
all_gsea_results.head()

In [ ]:
all_shap_results = pd.concat([v[1].assign(drug_id=k) for k, v in drug_results.items()])
all_shap_results.head()

In [ ]:
def get_cluster_orders(
    results_df: pd.DataFrame,
    value_col: str = "spearman_r",
) -> tuple[list[str], list[str]]:
    """Get hierarchically clustered orderings for gene sets and drugs.

    Parameters
    ----------
    results_df : pd.DataFrame
        Must have columns: 'set', 'drug_id', and value_col.
    value_col : str
        Column to use for clustering.

    Returns
    -------
    set_order : list[str]
        Gene set names in clustered order.
    drug_order : list[str]
        Drug names in clustered order.
    """
    mat = results_df.pivot_table(
        index="set", columns="drug_id", values=value_col
    ).fillna(0)

    row_order = mat.index.tolist()
    if len(mat) > 1:
        row_Z = linkage(pdist(mat.values, metric="euclidean"), method="average")
        row_order = mat.index[leaves_list(row_Z)].tolist()

    col_order = mat.columns.tolist()
    if len(mat.columns) > 1:
        col_Z = linkage(pdist(mat.values.T, metric="euclidean"), method="average")
        col_order = mat.columns[leaves_list(col_Z)].tolist()

    return row_order, col_order

In [ ]:
def beeswarm_offsets(
    df: pd.DataFrame, x_col: str, group_col: str, n_bins: int = 50
) -> pd.DataFrame:
    """Compute beeswarm y-offsets: bin x values within each group,
    then assign symmetric offsets based on rank within each bin."""
    records = []
    for group, gdf in df.groupby(group_col):
        # Bin the x values
        x = gdf[x_col].values
        bins = pd.cut(x, bins=n_bins, labels=False, include_lowest=True)
        gdf = gdf.copy()
        gdf["_bin"] = bins

        for b, bdf in gdf.groupby("_bin"):
            n = len(bdf)
            # Assign symmetric offsets: 0, -1, 1, -2, 2, ...
            offsets = []
            for i in range(n):
                if i == 0:
                    offsets.append(0)
                elif i % 2 == 1:
                    offsets.append((i + 1) // 2)
                else:
                    offsets.append(-i // 2)
            bdf = bdf.copy()
            bdf["_swarm_offset"] = offsets
            records.append(bdf)

    result = pd.concat(records)
    return result

In [ ]:
def summarize_gene_sets(df: pd.DataFrame, expression_z: pd.DataFrame) -> pd.DataFrame:
    results = []
    for (drug_id, gs), grp in df.groupby(["drug_id", "set"]):
        if gs not in expression_z.columns:
            continue
        merged = (
            grp.set_index("tumor_id")[["value"]]
            .join(
                expression_z[[gs]].rename(columns={gs: "z"}),
                how="inner",
            )
            .dropna()
        )
        if len(merged) < 5:
            continue

        r, p = stats.spearmanr(merged["z"], merged["value"])
        shap_var = merged["value"].var()

        results.append(
            {
                "drug_id": drug_id,
                "set": gs,
                "n_tumors": len(merged),
                "shap_var": shap_var,
                "spearman_r": r,
                "pval": p,
            }
        )

    res = pd.DataFrame(results)
    res["fdr"] = multipletests(res["pval"], method="fdr_bh")[1]
    return res.sort_values("shap_var", ascending=False)

In [ ]:
source = summarize_gene_sets(all_gsea_results, avg_gexp)
source["fdr"] = multipletests(source["pval"], method="fdr_bh")[1]
source["nl_fdr"] = -np.log10(source["fdr"])

set_order, drug_order = get_cluster_orders(source)

hmap = (
    alt.Chart(source, view=alt.ViewConfig(strokeOpacity=0, stroke="black"), width=700)
    .transform_filter(alt.datum.fdr < 0.05)
    .mark_circle(opacity=1)
    .encode(
        alt.X("set:N")
        .sort(set_order)
        .axis(labelLimit=500, grid=True, orient="top", labelAngle=-65, domainOpacity=1)
        .title(None),
        alt.Y("drug_id:N")
        .sort(drug_order)
        .axis(orient="left", domainOpacity=1)
        .title(None),
        alt.Color("spearman_r:Q")
        .scale(domain=(-1, 0, 1), scheme="redyellowblue")
        .title("Spearman"),
        alt.Size("nl_fdr:Q")
        .scale(range=(50, 300))
        .legend(values=(50, 100, 150))
        .title("-Log10(P-Value)"),
    )
)

In [ ]:
swarm_charts = []
for drug_id in ["Lapatinib", "Cisplatin"]:
    # filter to drugs and collapsed sets
    sets_to_plot = (
        source.query("drug_id == @drug_id").sort_values("pval")["set"].to_list()
    )

    source = (
        all_gsea_results.query("drug_id == @drug_id")
        .merge(
            avg_gexp.melt(ignore_index=False, var_name="set", value_name="z_gexp")
            .rename_axis(index="tumor_id")
            .reset_index(),
            on=["tumor_id", "set"],
        )
        .assign(sort_key=lambda df: df["z_gexp"].abs())
        .pipe(beeswarm_offsets, x_col="value", group_col="set", n_bins=30)
        .sort_values(["set", "sort_key"])
    )

    url = f"./temp/swarm_{drug_id}.json"
    source.to_json(url, orient="records")

    swarm_chart = (
        alt.Chart(url)
        .mark_circle(size=20, opacity=1)
        .encode(
            alt.Y("value:Q")
            .scale(domain=(-0.1, 0.1))
            .axis(grid=False, values=[-0.1, -0.5, 0.0, 0.5, 0.1])
            .title([f"{drug_id}: Aggregate SHAP Value", "(Sum Over Genes)"]),
            alt.X("set:N")
            .sort(set_order)
            .axis(orient="top", labelAngle=-65, grid=True, labelLimit=500, labels=False)
            .title(None),
            alt.Color("z_gexp:Q")
            .scale(scheme="redblue", reverse=True, domainMid=0)
            .legend(
                title="Z-Score Gene Expression", orient="right", gradientLength=200
            ),
            xOffset="_swarm_offset:Q",
            tooltip=["tumor_id:N", "set:N", "value:Q", "z_gexp:Q"],
        )
        .properties(height=200, width=700)
    )
    swarm_charts.append(swarm_chart)

In [ ]:
plot = (
    alt.vconcat(hmap, *swarm_charts)
    .resolve_scale(color="independent", size="independent")
    .resolve_scale(x="shared", color="independent")
)

configure_chart(plot)